# CitizenTech — Rota de menor "custo de dificuldade"
**Global Solution 2026.2 · Dynamic Programming · Equipe Winx**
André Rosa Ferreira Filho (RM567149) · Michel Lima Benchimol (RM567345) · Yuri Aroste Fukamachi (RM567314)

## 1. Problema
O *Kit de Autonomia no Transporte* ajuda pessoas cegas ou com baixa visão a se locomoverem sozinhas
pelo transporte público de São Paulo. Para recomendar um trajeto, o menor tempo não basta: esperar e
embarcar num ônibus, ou atravessar uma rua sem semáforo sonoro, é **mais difícil** do que caminhar
alguns minutos a mais. Precisamos do trajeto de **menor custo de dificuldade** entre dois pontos.

## 2. Fonte dos dados (simulada)
Os dados abaixo são **simulados**, inspirados no piloto do projeto: linha **271A-10**, **5 pontos**
e **2 travessias**. Os tempos são estimativas em minutos. Numa versão real, as listas viriam do GTFS da
SPTrans (pontos, linhas e horários) e do cadastro de semáforos com/sem sinal sonoro.
Os nomes dos pontos P2, P3 e P4 são genéricos porque os dados são fictícios.

**Custo de dificuldade** de um trecho = `tempo (min)` + `3` se for embarque no ônibus
+ `2` se atravessa rua com sinal sonoro + `10` se atravessa rua **sem** sinal sonoro.

In [1]:
# Parâmetros do custo de dificuldade (decisão do grupo, ajustáveis)
PESO_EMBARQUE = 3          # entrar no ônibus exige localizar a linha e a porta
PESO_TRAVESSIA_SONORA = 2  # atravessar com semáforo sonoro
PESO_TRAVESSIA_SEM_SOM = 10  # atravessar sem sinal sonoro (maior risco)

PONTOS = {
    "P1": "Av. Tenente Amaro",
    "P2": "Ponto 2",
    "P3": "Ponto 3",
    "P4": "Ponto 4",
    "P5": "Supermercado Priscilo",
}

# Cada trecho: (origem, destino, tempo_min, embarque, travessia)
# Nó "P1" = a pé no ponto 1. Nó "P1@bus" = dentro do ônibus, na altura do ponto 1.
TRECHOS = []

# Trechos a pé entre pontos vizinhos (valem nos dois sentidos).
# P2<->P3 tem travessia SEM som; P4<->P5 tem travessia COM som.
a_pe = [
    ("P1", "P2", 6, None),
    ("P2", "P3", 8, "sem_som"),
    ("P3", "P4", 6, None),
    ("P4", "P5", 7, "sonora"),
]
for u, v, t, trav in a_pe:
    TRECHOS.append((u, v, t, 0, trav))
    TRECHOS.append((v, u, t, 0, trav))

# Ônibus da linha 271A-10: só anda no sentido P1 -> P5.
viagens_onibus = [("P1", "P2", 4), ("P2", "P3", 5), ("P3", "P4", 4), ("P4", "P5", 6)]
for u, v, t in viagens_onibus:
    TRECHOS.append((u + "@bus", v + "@bus", t, 0, None))

# Embarque (espera + entrar) e desembarque em cada ponto
for p in PONTOS:
    TRECHOS.append((p, p + "@bus", 6, 1, None))   # 6 min de espera média
    TRECHOS.append((p + "@bus", p, 1, 0, None))   # 1 min para descer

print(len(TRECHOS), "trechos")

22 trechos


## 3. Modelagem
O mapa vira um **grafo orientado e ponderado**: cada nó é um lugar (a pé num ponto, ou dentro do ônibus
naquele ponto) e cada trecho é uma aresta com um custo. Guardamos o grafo numa **lista de adjacência**
(dicionário de listas) com o custo já calculado. Separar "a pé" de "no ônibus" em nós diferentes
permite cobrar o embarque só uma vez, e não a cada ponto percorrido dentro do ônibus.

In [2]:
def custo_trecho(tempo, embarque, travessia):
    """Custo de dificuldade de um trecho (ver seção 2)."""
    c = tempo + PESO_EMBARQUE * embarque
    if travessia == "sonora":
        c += PESO_TRAVESSIA_SONORA
    elif travessia == "sem_som":
        c += PESO_TRAVESSIA_SEM_SOM
    return c

def montar_grafo(trechos):
    """Lista de adjacência: grafo[u] = [(v, custo), ...]"""
    grafo = {}
    for u, v, t, emb, trav in trechos:
        grafo.setdefault(u, []).append((v, custo_trecho(t, emb, trav)))
        grafo.setdefault(v, [])
    return grafo

GRAFO = montar_grafo(TRECHOS)
print(len(GRAFO), "nós")
print("Saídas de P2:", GRAFO["P2"])

10 nós
Saídas de P2: [('P1', 6), ('P3', 18), ('P2@bus', 9)]


## 4. Algoritmo — programação dinâmica (Bellman-Ford)
**Subproblema:** `dp[k][v]` = menor custo para chegar em `v` partindo da origem usando **no máximo `k` trechos**.

**Recorrência:** `dp[k][v] = min( dp[k-1][v],  min sobre trechos (u→v) de dp[k-1][u] + custo(u,v) )`

**Caso base:** `dp[0][origem] = 0` e `dp[0][v] = ∞` para os demais.

Os subproblemas se repetem (o melhor caminho até `u` serve para vários `v`) e a solução ótima contém
soluções ótimas menores. Esses dois fatos caracterizam programação dinâmica. Como nenhum trajeto simples
passa de `V-1` trechos, a resposta está em `dp[V-1][destino]`. Para recuperar o caminho, guardamos de qual
nó viemos (`anterior`).

In [3]:
INF = float("inf")

def menor_custo(grafo, origem, destino):
    """Retorna (custo, caminho) de menor custo de dificuldade, ou (INF, []) se não há trajeto."""
    nos = list(grafo)
    dp = {v: INF for v in nos}            # dp atual (camada k)
    dp[origem] = 0                        # caso base
    anterior = {v: None for v in nos}

    for k in range(1, len(nos)):          # k = no máximo k trechos
        novo = dict(dp)                   # dp[k] começa igual a dp[k-1]
        mudou = False
        for u in nos:
            if dp[u] == INF:
                continue
            for v, c in grafo[u]:
                if dp[u] + c < novo[v]:   # recorrência
                    novo[v] = dp[u] + c
                    anterior[v] = u
                    mudou = True
        dp = novo
        if not mudou:                     # nada melhorou: já convergiu
            break

    if dp[destino] == INF:
        return INF, []
    caminho, v = [], destino
    while v is not None:                  # reconstrói o caminho de trás para frente
        caminho.append(v)
        v = anterior[v]
    return dp[destino], caminho[::-1]

## 5. Exemplos de entrada e saída
A função abaixo traduz o caminho técnico em instruções legíveis (o texto que o app falaria no fone).

In [4]:
def descrever(grafo, caminho):
    """Transforma a lista de nós em instruções passo a passo."""
    passos = []
    for u, v in zip(caminho, caminho[1:]):
        nome_u, nome_v = PONTOS[u.split("@")[0]], PONTOS[v.split("@")[0]]
        if v.endswith("@bus") and not u.endswith("@bus"):
            passos.append(f"Embarcar no ônibus 271A-10 no ponto {nome_u}")
        elif u.endswith("@bus") and not v.endswith("@bus"):
            passos.append(f"Descer no ponto {nome_u}")
        elif u.endswith("@bus"):
            continue  # seguindo dentro do ônibus
        else:
            trecho = next(x for x in TRECHOS if x[0] == u and x[1] == v)
            extra = {"sem_som": " (travessia SEM sinal sonoro)", "sonora": " (travessia com sinal sonoro)"}.get(trecho[4], "")
            passos.append(f"Caminhar de {nome_u} até {nome_v}{extra}")
    return passos

def rota(origem, destino):
    custo, caminho = menor_custo(GRAFO, origem, destino)
    print(f"{PONTOS[origem]} -> {PONTOS[destino]}   custo de dificuldade = {custo}")
    for i, p in enumerate(descrever(GRAFO, caminho), 1):
        print(f"  {i}. {p}")
    print()

rota("P1", "P5")   # trajeto completo
rota("P2", "P3")   # travessia sem som: compensa o ônibus?
rota("P3", "P4")   # trecho curto: a pé
rota("P4", "P5")   # travessia sonora: a pé
rota("P3", "P5")   # dois trechos a pé

Av. Tenente Amaro -> Supermercado Priscilo   custo de dificuldade = 29
  1. Embarcar no ônibus 271A-10 no ponto Av. Tenente Amaro
  2. Descer no ponto Supermercado Priscilo

Ponto 2 -> Ponto 3   custo de dificuldade = 15
  1. Embarcar no ônibus 271A-10 no ponto Ponto 2
  2. Descer no ponto Ponto 3

Ponto 3 -> Ponto 4   custo de dificuldade = 6
  1. Caminhar de Ponto 3 até Ponto 4

Ponto 4 -> Supermercado Priscilo   custo de dificuldade = 9
  1. Caminhar de Ponto 4 até Supermercado Priscilo (travessia com sinal sonoro)

Ponto 3 -> Supermercado Priscilo   custo de dificuldade = 15
  1. Caminhar de Ponto 3 até Ponto 4
  2. Caminhar de Ponto 4 até Supermercado Priscilo (travessia com sinal sonoro)



## 6. Testes de correção
1. **Valores conhecidos**, calculados à mão a partir da tabela de trechos.
2. **Comparação com força bruta:** para todos os pares de nós, o resultado do algoritmo precisa ser igual
   ao menor custo entre *todos* os caminhos simples (busca exaustiva).
3. **Casos de borda:** origem igual ao destino e destino inalcançável.

In [5]:
# 1) Valores conhecidos (conta feita à mão)
# P1->P5 de ônibus: embarque 6+3=9, viagem 4+5+4+6=19, desembarque 1 -> 29
assert menor_custo(GRAFO, "P1", "P5")[0] == 29
# P3->P4 a pé: 6 (sem travessia)
assert menor_custo(GRAFO, "P3", "P4") == (6, ["P3", "P4"])
# P4->P5 a pé: 7 + 2 (travessia sonora) = 9, melhor que ônibus (9+6+1 = 16)
assert menor_custo(GRAFO, "P4", "P5") == (9, ["P4", "P5"])
# P2->P3: a pé = 8+10 = 18; ônibus = 9+5+1 = 15  -> o ônibus vence por causa da travessia sem som
assert menor_custo(GRAFO, "P2", "P3")[0] == 15
print("1) valores conhecidos: OK")

# 2) Força bruta: testa todos os caminhos simples
def forca_bruta(grafo, origem, destino):
    melhor = INF
    def dfs(u, visitados, acumulado):
        nonlocal melhor
        if u == destino:
            melhor = min(melhor, acumulado)
            return
        for v, c in grafo[u]:
            if v not in visitados:
                dfs(v, visitados | {v}, acumulado + c)
    dfs(origem, {origem}, 0)
    return melhor

pares = 0
for o in GRAFO:
    for d in GRAFO:
        assert menor_custo(GRAFO, o, d)[0] == forca_bruta(GRAFO, o, d), (o, d)
        pares += 1
print(f"2) força bruta: OK em {pares} pares origem/destino")

# 3) Bordas
assert menor_custo(GRAFO, "P3", "P3") == (0, ["P3"])
isolado = {"A": [], "B": []}
assert menor_custo(isolado, "A", "B") == (INF, [])
# o ônibus só anda no sentido P1 -> P5: de P5 de volta a P1 só dá a pé
assert menor_custo(GRAFO, "P5", "P1")[1] == ["P5", "P4", "P3", "P2", "P1"]
print("3) casos de borda: OK")

1) valores conhecidos: OK
2) força bruta: OK em 100 pares origem/destino
3) casos de borda: OK


## 7. Eficiência
Seja `V` o número de nós e `E` o número de trechos.

| | Bellman-Ford (DP) | Força bruta (todos os caminhos) |
|---|---|---|
| Tempo | **O(V · E)** — até `V-1` camadas, cada uma varre as `E` arestas | exponencial no pior caso |
| Memória | **O(V)** — guardamos só a camada atual e o vetor `anterior` | O(V) de recursão |

A medição abaixo usa grafos sintéticos em linha com `n` nós (cada nó liga ao seguinte e ao anterior)
para mostrar o crescimento do tempo.

In [6]:
import time

def grafo_linha(n):
    g = {i: [] for i in range(n)}
    for i in range(n - 1):
        g[i].append((i + 1, 1))
        g[i + 1].append((i, 1))
    return g

print(f"{'n':>6} {'E':>6} {'tempo (s)':>10}")
for n in (50, 100, 200, 400):
    g = grafo_linha(n)
    e = sum(len(x) for x in g.values())
    ini = time.perf_counter()
    custo, _ = menor_custo(g, 0, n - 1)
    dur = time.perf_counter() - ini
    assert custo == n - 1
    print(f"{n:>6} {e:>6} {dur:>10.4f}")

     n      E  tempo (s)
    50     98     0.0003
   100    198     0.0009
   200    398     0.0036
   400    798     0.0144


**Leitura:** com `V` e `E` dobrando juntos, o tempo cresce de forma aproximadamente quadrática, como
prevê O(V·E). Para o piloto (10 nós e 22 trechos) a resposta é instantânea.

**Limitações e próximos passos:** o algoritmo funciona com custos negativos (Dijkstra não), mas como nossos
custos são sempre positivos, `Dijkstra` com fila de prioridade, O(E log V), seria mais rápido numa cidade
inteira. Também faltaria usar horários reais dos ônibus em vez da espera média de 6 minutos.